<a href="https://colab.research.google.com/github/FatiBuuloloo/Image_Anomaly_Detection_on_MVTec_Dataset-mini_project_005/blob/main/Image_anomaly_detection_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install anomalib

In [ ]:
!pip install onnxscript

In [ ]:
from anomalib.data import MVTec
from anomalib.models import Padim
from anomalib.engine import Engine
from anomalib.data import Folder
from anomalib.models import Patchcore
import matplotlib.pyplot as plt

In [ ]:
def denormalize(img_tensor):
    """Mengubah tensor menjadi gambar yang bisa dilihat mata manusia"""
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    # Normalisasi standar ImageNet biasanya perlu di-undo agar warna kembali normal
    # Ini cara simpel (min-max scaling) agar gambar tampil cerah
    img = (img - img.min()) / (img.max() - img.min())
    return img
def show_results(predictions, num_samples=5):
    """Menampilkan Gambar Asli, Heatmap, dan Info Prediksi dengan nama field terbaru"""

    for i, pred in enumerate(predictions):
        if i >= num_samples: break

        # Perubahan kunci: 'anomaly_map' (tunggal) bukan 'anomaly_maps'
        # Dan 'pred_score' bukan 'pred_scores'
        try:
            anomaly_map = pred.anomaly_map[0].cpu().numpy()
            score = pred.pred_score[0].item()
            label_idx = pred.pred_label[0].item()
            img_original = denormalize(pred.image[0])
        except AttributeError:
            # Jika masih error menggunakan dot notation, coba gunakan dictionary notation
            anomaly_map = pred['anomaly_map'][0].cpu().numpy()
            score = pred['pred_score'][0].item()
            label_idx = pred['pred_label'][0].item()
            img_original = denormalize(pred['image'][0])

        label = "ANOMALI (CACAT)" if label_idx else "NORMAL (BAGUS)"

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # 1. Gambar Asli
        axes[0].imshow(img_original)
        axes[0].set_title(f"Original\n{label}", color='red' if label_idx else 'green')
        axes[0].axis('off')

        # 2. Anomaly Map
        im = axes[1].imshow(anomaly_map, cmap='jet')
        axes[1].set_title(f"Heatmap\nScore: {score:.4f}")
        axes[1].axis('off')
        plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

        # 3. Overlay
        axes[2].imshow(img_original)
        axes[2].imshow(anomaly_map, cmap='jet', alpha=0.4)
        axes[2].set_title("Overlay Location")
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

# 5. Tampilkan Hasil

# Iterative ZIP Gdrive

In [ ]:
categories = [
    "bottle", "grid","cable","capsule", "carpet",
    "leather", "metal_nut", "pill", "screw", "tile",
   "toothbrush", "transistor", "wood", "zipper"
,"hazelnut"]

In [ ]:
!ls /content/drive/MyDrive/dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
all_data = {}

for item in categories:
    path_zip = f"/content/drive/MyDrive/dataset/{item}.zip"
    print(f"Unzipping {item}")
    !unzip -oq "$path_zip" -d "/content/dataset/"
    dm = Folder(
        name=f"{item}_project",
        root=f"/content/dataset/{item}",
        normal_dir="train",
        abnormal_dir="test",
        normal_test_dir="test/good",
        #mask_dir="ground_truth",
        #task="segmentation",
        #image_size=(256, 256),
        #train_batch_size=32,
        eval_batch_size=1)
    all_data[item] = dm
    print(f"finish unzipping {item}")
    print()

# Toothbrush

In [ ]:
data_toothbrush = all_data["toothbrush"]
data_toothbrush.setup()

In [ ]:
model_toothbrush = Padim(backbone="resnet18")

In [ ]:
engine_toothbrush = Engine()
engine_toothbrush.fit(model=model_toothbrush, datamodule=data_toothbrush)

In [ ]:
test_toothbrush = engine_toothbrush.test(model=model_toothbrush, datamodule=data_toothbrush)
print(test_toothbrush)

In [ ]:
predictions_toothbrush = engine_toothbrush.predict(model=model_toothbrush,datamodule=data_toothbrush)
show_results(predictions_toothbrush,num_samples=5)

# Wood

In [ ]:
data_wood = all_data["wood"]
data_wood.setup()

In [ ]:
model_wood = Padim(backbone="resnet18")

In [ ]:
engine_wood = Engine()
engine_wood.fit(model=model_wood, datamodule=data_wood)

In [ ]:
test_wood = engine_wood.test(model=model_wood, datamodule=data_wood)
print(test_wood)

In [ ]:
predictions_wood = engine_wood.predict(model=model_wood,datamodule=data_wood)
show_results(predictions_wood,num_samples=5)

# Zipper

In [ ]:
data_zipper = all_data["zipper"]
data_zipper.setup()

In [ ]:
model_zipper = Padim(backbone="resnet18")

In [ ]:
engine_zipper = Engine()
engine_zipper.fit(model=model_zipper, datamodule=data_zipper)

In [ ]:
test_zipper = engine_zipper.test(model=model_zipper, datamodule=data_zipper)
print(test_zipper)

In [ ]:
predictions_zipper = engine_zipper.predict(model=model_zipper,datamodule=data_zipper)
show_results(predictions_zipper,num_samples=5)

patchcore model

In [ ]:
model_zipper_2 = Patchcore(
    backbone="wide_resnet50_2",
    pre_trained=True,
    coreset_sampling_ratio=0.01, # Rasio memori coreset
)

engine_zipper_2 = Engine(
    enable_checkpointing=True, # Matikan sementara untuk cek error
    logger=True,
enable_progress_bar=False,               # Matikan logger agar tidak konflik dengan tampilan Colab
    devices=1
)
engine_zipper_2.fit(model=model_zipper_2, datamodule=data_zipper)

In [ ]:
test_zipper_2 = engine_zipper_2.test(model=model_zipper_2, datamodule=data_zipper)
print(test_zipper_2)

In [ ]:
predictions_zipper_2 = engine_zipper_2.predict(model=model_zipper_2,datamodule=data_zipper)
show_results(predictions_zipper_2,num_samples=5)

# Hazelnut

In [ ]:
data_hazelnut = all_data["hazelnut"]
data_hazelnut.setup()

In [ ]:
model_hazelnut = Padim(backbone="resnet18")

In [ ]:
engine_hazelnut = Engine()
engine_hazelnut.fit(model=model_hazelnut, datamodule=data_hazelnut)

In [ ]:
test_hazelnut = engine_hazelnut.test(model=model_hazelnut, datamodule=data_hazelnut)
print(test_hazelnut)

In [ ]:
predictions_hazelnut = engine_hazelnut.predict(model=model_hazelnut,datamodule=data_hazelnut)
show_results(predictions_hazelnut,num_samples=5)

patchcore model

In [ ]:
model_hazelnut_2 = Patchcore(
    backbone="wide_resnet50_2",
    pre_trained=True,
    coreset_sampling_ratio=0.01, # Rasio memori coreset
)

engine_hazelnut_2 = Engine(
    enable_checkpointing=True, # Matikan sementara untuk cek error
    logger=True,
enable_progress_bar=False,               # Matikan logger agar tidak konflik dengan tampilan Colab
    devices=1
)
engine_hazelnut_2.fit(model=model_hazelnut_2, datamodule=data_hazelnut)

In [ ]:
test_hazelnut_2 = engine_hazelnut_2.test(model=model_hazelnut_2, datamodule=data_hazelnut)
print(test_hazelnut_2)

In [ ]:
predictions_hazelnut_2 = engine_hazelnut_2.predict(model=model_hazelnut_2,datamodule=data_hazelnut)
show_results(predictions_hazelnut_2,num_samples=5)